# Breast Histology Specialist — IDC Classifier (CliniqAI)

**Dataset:** [Breast Histopathology Images (IDC)](https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images) — Paul Mooney (Kaggle)

- 277,524 image patches, 50x50 px, extracted from 162 whole-slide images
- 279 unique patients
- Binary labels: `0` = IDC-negative (non-cancerous), `1` = IDC-positive (Invasive Ductal Carcinoma)
- Class split: ~198,738 negative (71.6%) / ~78,786 positive (28.4%) - imbalanced, handled via class-weighted loss

**Split strategy:** patient-level (not patch-level). Patches from the same patient are visually
correlated, so a random patch split leaks information across train/val/test and inflates accuracy.
We split by patient ID: 70% train / 15% val / 15% test.

**Config (matches your other 3 specialists):** EfficientNet-B0, batch size 32, lr 1e-3, weight
decay 1e-4, early stopping patience 10, 224x224 input, `num_workers=0` (Windows).


## 1. Session setup

Run this first in every session -- sets working directory and import path so the rest of your `CliniqAI` modules resolve correctly.

In [ ]:
import os
import sys

# Adjust if your repo lives elsewhere
PROJECT_ROOT = r"E:\programming\CliniqAI"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

## 2. Download the dataset

Uses your existing Kaggle CLI config at `C:\Users\anish\.kaggle\kaggle.json`. The zip is
~3.1 GB, unzipped ~5.8 GB -- make sure you have disk space before running this cell.

This only needs to run once. If `data/raw/breast_histology` already has content, skip this cell.

In [ ]:
import subprocess
from pathlib import Path

DATA_DIR = Path("data/raw/breast_histology")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not any(DATA_DIR.iterdir()):
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", "paultimothymooney/breast-histopathology-images",
        "-p", str(DATA_DIR),
        "--unzip"
    ], check=True)
    print("Download complete.")
else:
    print("Data already present, skipping download.")

## 3. Inspect folder structure

The dataset unzips into per-patient folders, each containing `0/` (non-IDC) and `1/` (IDC) subfolders
of 50x50 PNG patches:

```
data/raw/breast_histology/
    <patient_id>/
        0/
            <patient_id>_idx5_x###_y###_class0.png
        1/
            <patient_id>_idx5_x###_y###_class1.png
```

We won't reorganize/copy these (277K small files, no need to duplicate ~5.8 GB) -- the Dataset class
below reads directly from this structure.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/raw/breast_histology")

# Some Kaggle mirrors nest one extra folder level (e.g. IDC_regular_ps50_idx5/). Detect it.
nested = DATA_DIR / "IDC_regular_ps50_idx5"
ROOT = nested if nested.exists() else DATA_DIR

patient_dirs = sorted([p for p in ROOT.iterdir() if p.is_dir() and p.name.isdigit()])
print(f"Root: {ROOT}")
print(f"Found {len(patient_dirs)} patient folders")
print("Example:", patient_dirs[0].name if patient_dirs else "none found")

## 4. Patient-level train/val/test split (70/15/15)

In [ ]:
import random

random.seed(42)
patient_ids = [p.name for p in patient_dirs]
random.shuffle(patient_ids)

n = len(patient_ids)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_patients = set(patient_ids[:n_train])
val_patients = set(patient_ids[n_train:n_train + n_val])
test_patients = set(patient_ids[n_train + n_val:])

print(f"Patients -> train: {len(train_patients)}, val: {len(val_patients)}, test: {len(test_patients)}")

## 5. Build (path, label) index for each split

Scans every patient folder once and buckets file paths by split + class.

In [ ]:
def collect_samples(patient_set):
    samples = []
    for pdir in patient_dirs:
        if pdir.name not in patient_set:
            continue
        for label in (0, 1):
            class_dir = pdir / str(label)
            if not class_dir.exists():
                continue
            for f in class_dir.glob("*.png"):
                samples.append((str(f), label))
    return samples

train_samples = collect_samples(train_patients)
val_samples = collect_samples(val_patients)
test_samples = collect_samples(test_patients)

print(f"Train: {len(train_samples)} patches")
print(f"Val:   {len(val_samples)} patches")
print(f"Test:  {len(test_samples)} patches")

train_pos = sum(1 for _, l in train_samples if l == 1)
print(f"Train class balance -> negative: {len(train_samples) - train_pos}, positive: {train_pos}")

## 6. Dataset, transforms, dataloaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32

class IDCDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = IDCDataset(train_samples, transform=train_transform)
val_dataset = IDCDataset(val_samples, transform=eval_transform)
test_dataset = IDCDataset(test_samples, transform=eval_transform)

# num_workers=0 required on Windows
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches -> train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

## 7. Model -- EfficientNet-B0 (binary head)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)  # IDC-negative, IDC-positive
model = model.to(device)

print(model.classifier)

## 8. Loss, optimizer, class weighting

Class weights offset the ~71.6% / 28.4% imbalance so the model doesn't just learn to
predict "negative" every time.

In [ ]:
import numpy as np

neg_count = len(train_samples) - train_pos
pos_count = train_pos
total = neg_count + pos_count

# Inverse-frequency weighting
weight_neg = total / (2.0 * neg_count)
weight_pos = total / (2.0 * pos_count)
class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32).to(device)

print(f"Class weights -> negative: {weight_neg:.3f}, positive: {weight_pos:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

## 9. Training loop with early stopping (patience=10)

In [ ]:
import time
from pathlib import Path

CHECKPOINT_DIR = Path("models/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "breast_histology_efficientnet_b0.pt"

NUM_EPOCHS = 30
PATIENCE = 10

best_val_loss = float("inf")
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train_mode)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train_mode:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if train_mode:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
          f"{elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "epoch": epoch,
            "class_names": ["idc_negative", "idc_positive"],
        }, CHECKPOINT_PATH)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs (no improvement for {PATIENCE} epochs).")
            break

print("\nTraining complete. Best val_loss:", best_val_loss)

## 10. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Load best checkpoint and evaluate on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

print("Test accuracy:", (np.array(all_preds) == np.array(all_labels)).mean())
print("Test ROC-AUC:", roc_auc_score(all_labels, all_probs))
print()
print(classification_report(all_labels, all_preds, target_names=["idc_negative", "idc_positive"]))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["idc_negative", "idc_positive"],
            yticklabels=["idc_negative", "idc_positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Breast Histology (IDC) -- Confusion Matrix")
plt.show()

## Notes

- **Checkpoint saved to:** `models/checkpoints/breast_histology_efficientnet_b0.pt` -- sync to
  Google Drive (`medical-ai-models/trained_models/`) same as your other 3 specialists.
- **Patient-level split** means val/test accuracy here is a more honest estimate of real-world
  performance than a patch-level split would give you -- expect this number to look slightly lower
  than your retinal/kidney/brain specialists, and that's expected/correct, not a bug.
- If VRAM runs out at batch size 32 on the RTX 4050 (6GB), drop to 16 and it'll still train fine --
  277K images means plenty of steps per epoch either way.
- This slots in as a new Stage 2 modality class (`breast_histology`) once you're ready to retrain
  the router's Stage 2 classifier alongside Roneet's modalities.
